# ⚡ BoneRAG — Free Google Colab GPU Backend (pyngrok)

### 📌 Hướng dẫn:
1. **Bật GPU**: `Runtime` → `Change runtime type` → `T4 GPU`
2. Điền **ngrok authtoken** của bạn vào ô `NGROK_TOKEN` ở Step 3 (lấy miễn phí tại https://dashboard.ngrok.com/authtokens)
3. Bấm **Runtime → Run all (Ctrl+F9)**
4. Copy link `https://xxxx.ngrok-free.app` → dán vào Vercel `VITE_API_BASE_URL` → Redeploy

In [ ]:
# [Step 1] Kiểm tra GPU & Cài đặt thư viện
!nvidia-smi
!pip install -q torch torchvision transformers open_clip_torch faiss-cpu pillow numpy huggingface_hub pyngrok

In [ ]:
# [Step 2] Tải mã nguồn BoneRAG mới nhất từ GitHub
import os
if not os.path.exists('/content/boneRAG'):
    !git clone https://github.com/thanhnghi-do-2k3/boneRAG.git /content/boneRAG
else:
    %cd /content/boneRAG && !git pull
%cd /content/boneRAG

In [ ]:
# [Step 3] ⚙️ ĐIỀN NGROK TOKEN VÀO ĐÂY (lấy miễn phí tại https://dashboard.ngrok.com/authtokens)
NGROK_TOKEN = "REDACTED"  # <-- Dán token ngrok của bạn vào đây

import subprocess, time, urllib.request, urllib.error
from pyngrok import ngrok, conf

PORT = 8088

# Cài ngrok token
if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN
else:
    print("⚠️ Chưa điền NGROK_TOKEN! Lấy miễn phí tại: https://dashboard.ngrok.com/authtokens")
    raise ValueError("Cần NGROK_TOKEN để tiếp tục.")

# Khởi động BoneRAG server ngầm
server_proc = subprocess.Popen(
    ['python3', 'demo-app/server.py', '--host', '0.0.0.0', '--port', str(PORT)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

print('⏳ Đang chờ BoneRAG server khởi động...')
for i in range(40):
    time.sleep(3)
    try:
        urllib.request.urlopen(f'http://127.0.0.1:{PORT}/api/records')
        print(f'✅ Server đã sẵn sàng! (sau {(i+1)*3}s)')
        break
    except Exception:
        print(f'   [{i+1}/40] Đang chờ server...', end='\r')
else:
    out, _ = server_proc.communicate(timeout=2)
    print('\n❌ Server lỗi! Log:')
    print(out.decode('utf-8', errors='replace')[:3000])
    raise RuntimeError('Server không khởi động được.')

# Mở ngrok tunnel
tunnel = ngrok.connect(PORT, bind_tls=True)
PUBLIC_URL = tunnel.public_url

print()
print('=' * 60)
print('🚀 BACKEND GPU ĐANG CHẠY — COPY LINK NÀY VÀO VERCEL:')
print('=' * 60)
print(f'  VITE_API_BASE_URL = {PUBLIC_URL}')
print('=' * 60)
print('📌 Sau khi dán vào Vercel → bấm Save → bấm Redeploy!')
print('⚠️  Giữ ô này đang chạy. Đừng Interrupt hoặc đóng Colab!')